<a href="https://colab.research.google.com/github/brandonwolk/Kaneohe-Bay-Coral-Restoration-GIS/blob/main/capstone/02_MCDA_Spatial_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# MODULE 2: MULTI-CRITERIA DECISION ANALYSIS (MCDA) & SPATIAL MODELING
# Kāneʻohe Bay Coral Restoration Spatial Framework
# ==============================================================================

import os
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree
from pyproj import Transformer
from google.colab import drive

# --- PROACTIVE MOUNT POINT CLEANSE & SMART DRIVE MOUNT ---
mount_point = '/content/drive'
try:
    drive.flush_and_unmount()
except:
    pass

if os.path.exists(mount_point):
    try:
        shutil.rmtree(mount_point)
    except Exception as e:
        print(f"Clearing mountpoint warning: {e}")

os.makedirs(mount_point, exist_ok=True)
drive.mount(mount_point, force_remount=True)

CAPSTONE_ROOT = "/content/drive/MyDrive/Capstone"
BATHY_PATH        = f"{CAPSTONE_ROOT}/02 Processed Rasters/Kaneohe_Bathy_Final.tif"
DHW_HIST_PATH     = f"{CAPSTONE_ROOT}/02 Processed Rasters/kaneohe_bay_CRW_DHW_2000_2025_max.tif"
DHW_PREDICT_PATH  = f"{CAPSTONE_ROOT}/03 Predictive Rasters/kaneohe_bay_MIROC6_DHW_SSP585_2025_2050.tif"
WQ_CSV_PATH       = f"{CAPSTONE_ROOT}/01 Raw Data/PacIOOS_CWB_WaterQuality.csv"
OUTPUT_DIR        = f"{CAPSTONE_ROOT}/04 MCDA Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify foundational path exists
if not os.path.exists(BATHY_PATH):
    raise FileNotFoundError(f"[ERROR] Master bathymetry raster not found at: {BATHY_PATH}")

# Master Grid Lock from Bathymetry
with rasterio.open(BATHY_PATH) as src:
    master_profile   = src.profile.copy()
    master_crs       = src.crs
    master_transform = src.transform
    master_width     = src.width
    master_height    = src.height
    master_shape     = (master_height, master_width)
    master_res       = src.res
    bathy_array      = src.read(1, masked=True)

bathy_filled = bathy_array.filled(np.nan) if np.ma.isMaskedArray(bathy_array) else bathy_array

# Water Quality IDW Interpolation Pipeline
TRUE_BAY_STATION_IDS = [190, 191, 208, 241, 242, 260, 261, 281, 362]
wq_raw = pd.read_csv(WQ_CSV_PATH, skiprows=[1])
wq_raw["longitude"] = pd.to_numeric(wq_raw["longitude"], errors="coerce")
wq_raw["latitude"]  = pd.to_numeric(wq_raw["latitude"], errors="coerce")
wq_raw["turbidity"] = pd.to_numeric(wq_raw["turbidity"], errors="coerce")
wq_raw["salinity"]  = pd.to_numeric(wq_raw["salinity"], errors="coerce")
wq_bay = wq_raw[wq_raw["location_id"].isin(TRUE_BAY_STATION_IDS)].dropna(subset=["longitude", "latitude"])

transformer = Transformer.from_crs("EPSG:4326", master_crs, always_xy=True)
cols_idx, rows_idx = np.meshgrid(np.arange(master_width), np.arange(master_height))
grid_x = master_transform.c + (cols_idx + 0.5) * master_transform.a
grid_y = master_transform.f + (rows_idx + 0.5) * master_transform.e

print("MCDA Spatial Modeling Environment Initialized Successfully.")

Mounted at /content/drive
MCDA Spatial Modeling Environment Initialized Successfully.
